# **Data Cleaning**

In [1]:
# Load pandas for data processing and datetime utilities for any date operations.
import pandas as pd

# Set option to display all columns and format float values to two decimal places (to avoid scientific notation).
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.2f}'.format)

In [2]:
# Load the raw e-commerce dataset for cleaning.
ecommerce_df = pd.read_csv(r'dataset/raw/ecommerce_dataset_+1m.csv')

## Initial Cleaning

Goals for this section:
- Remove unnecessary / redundant columns
- Inspect data quality (nulls, sample, dtypes)
- Round float columns to 2 decimal places
- Flag logically invalid values

In [3]:
# Display all the columns
ecommerce_df.columns

Index(['order_id', 'order_date', 'order_year', 'order_month', 'order_day',
       'order_hour', 'order_minute', 'order_second', 'is_weekend',
       'order_status', 'return_reason', 'customer_id', 'customer_name',
       'gender', 'age', 'customer_segment', 'country', 'city',
       'customer_loyalty_score', 'total_orders_by_customer',
       'account_creation_date', 'product_id', 'product_name', 'category',
       'sub_category', 'brand', 'product_rating_avg', 'product_reviews_count',
       'stock_quantity', 'unit_price_usd', 'quantity', 'discount_percent',
       'discount_amount_usd', 'total_price_usd', 'cost_usd', 'profit_usd',
       'tax_usd', 'currency', 'payment_method', 'payment_status',
       'installment_plan', 'shipping_method', 'shipping_cost_usd',
       'delivery_days', 'shipping_country', 'warehouse_location',
       'delivery_status', 'rating', 'review_sentiment', 'customer_feedback',
       'coupon_used', 'coupon_code', 'campaign_source', 'device_type',
       'traf

### 1. Remove unnecessary columns

In [4]:
# Eliminate columns that are not relevant to the analysis or contain redundant information.

ecommerce_df = ecommerce_df[
    [
        # TIME & CONTEXT    
        'order_date',
        'order_year',
        'order_month',
        'is_weekend',
        
        # CUSTOMER DEMOGRAPHICS / GEOGRAPHY
        'customer_name',
        'gender',
        'age',
        'customer_segment',
        'country',
        
        # PRODUCT
        'order_status',
        'category',
        'sub_category',
        'unit_price_usd',
        'quantity',
        
        # FINANCIALS
        'discount_percent',
        'total_price_usd',
        'profit_usd',
        'profit_margin_percent',
        
        # PAYMENT & SHIPPING
        'payment_method',
        'shipping_method',
        'shipping_cost_usd',
        'delivery_days',
        'shipping_country',
        
        # CUSTOMER BEHAVIOR
        'rating',
        'customer_loyalty_score',
        'coupon_used',
        'session_duration_minutes',
        'pages_visited',
        'abandoned_cart_before',
        
        # RISK & PERFORMANCE
        'fraud_risk_score',
        'device_type',
        
        # MARKETING
        'campaign_source',
        'traffic_source',
    ]
].copy()

### 2. Rename Columns

In [5]:
ecommerce_df = ecommerce_df.rename(
    columns={
        'total_price_usd' : 'revenue_usd',
        'session_duration_minutes' : 'session_duration_min'
    }
)

### 3. Data quality check

In [6]:
ecommerce_df.info()  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000123 entries, 0 to 1000122
Data columns (total 33 columns):
 #   Column                  Non-Null Count    Dtype  
---  ------                  --------------    -----  
 0   order_date              1000123 non-null  object 
 1   order_year              1000123 non-null  int64  
 2   order_month             1000123 non-null  int64  
 3   is_weekend              1000123 non-null  object 
 4   customer_name           1000123 non-null  object 
 5   gender                  1000123 non-null  object 
 6   age                     1000123 non-null  int64  
 7   customer_segment        1000123 non-null  object 
 8   country                 1000123 non-null  object 
 9   order_status            1000123 non-null  object 
 10  category                1000123 non-null  object 
 11  sub_category            1000123 non-null  object 
 12  unit_price_usd          1000123 non-null  float64
 13  quantity                1000123 non-null  int64  
 14  di

In [7]:
# Missing values (%)
print(((ecommerce_df.isnull().sum() / len(ecommerce_df)) * 100).round(2).to_string())

order_date               0.00
order_year               0.00
order_month              0.00
is_weekend               0.00
customer_name            0.00
gender                   0.00
age                      0.00
customer_segment         0.00
country                  0.00
order_status             0.00
category                 0.00
sub_category             0.00
unit_price_usd           0.00
quantity                 0.00
discount_percent         0.00
revenue_usd              0.00
profit_usd               0.00
profit_margin_percent    0.00
payment_method           0.00
shipping_method          0.00
shipping_cost_usd        0.00
delivery_days            0.00
shipping_country         0.00
rating                   0.00
customer_loyalty_score   0.00
coupon_used              0.00
session_duration_min     0.00
pages_visited            0.00
abandoned_cart_before    0.00
fraud_risk_score         0.00
device_type              0.00
campaign_source          0.00
traffic_source           0.00


In [8]:
# Random sample
ecommerce_df.sample(10)

,order_date,order_year,order_month,is_weekend,customer_name,gender,age,customer_segment,country,order_status,category,sub_category,unit_price_usd,quantity,discount_percent,revenue_usd,profit_usd,profit_margin_percent,payment_method,shipping_method,shipping_cost_usd,delivery_days,shipping_country,rating,customer_loyalty_score,coupon_used,session_duration_min,pages_visited,abandoned_cart_before,fraud_risk_score,device_type,campaign_source,traffic_source
81437,2024-11-29 22:14:51.198564,2024,11,No,Erin Nguyen,Female,47,Premium,Spain,Completed,Health,Medical Devices,184.72,1,0,184.72,93.89,50.83,Apple Pay,Standard,4.31,12,Spain,2,44.20,Yes,38.90,11,No,22.10,Desktop,Organic,Social
491848,2025-06-12 17:39:20.214690,2025,6,No,Natalie Perez,Female,23,Premium,Belgium,Processing,Health,Medical Devices,154.67,5,10,696.02,203.92,29.30,Credit Card,Standard,7.69,5,Belgium,2,7.50,No,51.00,10,Yes,75.30,Desktop,Facebook,Social
974130,2025-09-14 00:37:10.441096,2025,9,Yes,Jordan Lee,Male,32,Regular,France,Processing,Home,Appliances,179.23,5,15,761.73,235.53,30.92,Bank Transfer,Next Day,8.46,12,France,2,34.80,Yes,2.50,9,Yes,49.50,Desktop,Email,Email
135634,2025-12-22 00:20:34.146042,2025,12,No,Breanna Fowler,Female,64,Premium,Italy,Cancelled,Sports,Sports Wear,196.35,1,15,166.90,39.88,23.89,Apple Pay,Next Day,10.12,10,Italy,2,77.20,Yes,13.90,10,Yes,74.10,Mobile,Organic,Social
209234,2025-07-20 22:37:07.079270,2025,7,Yes,Edwin Tucker,Male,41,Regular,Spain,Returned,Home,Bedding,150.18,2,5,285.34,128.22,44.94,Apple Pay,Standard,1.95,13,Spain,5,9.90,Yes,40.90,2,Yes,77.00,Desktop,Organic,Direct
289178,2025-02-13 04:32:23.408689,2025,2,No,Deborah Cunningham,Female,36,Premium,Spain,Completed,Electronics,Laptops,51.56,5,0,257.80,104.70,40.61,Bank Transfer,Economy,19.65,14,Spain,5,82.70,Yes,24.90,10,Yes,78.90,Desktop,Email,Search
545042,2024-08-31 16:41:47.409197,2024,8,Yes,Christopher Blair,Male,70,Regular,Australia,Completed,Home,Bedding,134.40,4,10,483.84,181.88,37.59,Apple Pay,Standard,24.02,5,Australia,2,29.50,No,32.70,14,No,99.20,Desktop,Instagram,Referral
567880,2025-05-03 17:12:38.858228,2025,5,Yes,Gregory Berry,Male,58,Regular,Australia,Completed,Electronics,Laptops,226.26,4,0,905.04,353.56,39.07,Credit Card,Express,19.93,10,Australia,1,73.50,No,42.10,17,No,54.70,Desktop,Facebook,Social
113236,2025-05-17 10:29:08.676013,2025,5,Yes,Crystal Barker,Female,39,Premium,United Kingdom,Completed,Clothing,Womens Wear,75.92,1,20,60.74,20.59,33.90,Debit Card,Express,9.89,9,United Kingdom,2,26.50,No,25.90,20,No,0.50,Tablet,Google Ads,Direct
212357,2024-04-29 00:02:36.642396,2024,4,No,Jennifer Conner,Female,24,Regular,Canada,Completed,Sports,Accessories,121.91,5,0,609.55,265.35,43.53,Bank Transfer,Standard,4.85,2,Canada,5,16.30,Yes,10.50,3,Yes,85.60,Tablet,Google Ads,Email


### 4. Round float columns to 2 decimal places

In [9]:
# Round float values to two decimal places for cleaner output.
float_cols = ecommerce_df.select_dtypes('float')

for col in float_cols.columns:
    ecommerce_df[col] = ecommerce_df[col].round(2)

### 5. Flag logically invalid values

In [10]:
# 1. Age
invalid_age = ecommerce_df[(ecommerce_df['age'] < 0) | (ecommerce_df['age'] > 120)]

# 2. Quantity
invalid_quantity = ecommerce_df[ecommerce_df['quantity'] <= 0]

# 3. Discount
invalid_discount = ecommerce_df[
    (ecommerce_df['discount_percent'] < 0) | (ecommerce_df['discount_percent'] > 100)
]

# 4. Rating (fix this!)
invalid_rating = ecommerce_df[(ecommerce_df['rating'] < 1) | (ecommerce_df['rating'] > 5)]

# 5. Delivery days
invalid_delivery = ecommerce_df[
    (ecommerce_df['delivery_days'] < 0) | (ecommerce_df['delivery_days'] > 60)
]

# 6. Monetary values
invalid_money = ecommerce_df[
    (ecommerce_df['unit_price_usd'] < 0) |
    (ecommerce_df['revenue_usd'] < 0) |
    (ecommerce_df['shipping_cost_usd'] < 0)
]

# 7. Fraud score
invalid_fraud = ecommerce_df[
    (ecommerce_df['fraud_risk_score'] < 0) | (ecommerce_df['fraud_risk_score'] > 100)
]

# Print summary
print(f'Invalid Age        : {len(invalid_age)}')
print(f'Invalid Quantity   : {len(invalid_quantity)}')
print(f'Invalid Discount   : {len(invalid_discount)}')
print(f'Invalid Rating     : {len(invalid_rating)}')
print(f'Invalid Delivery   : {len(invalid_delivery)}')
print(f'Invalid Money      : {len(invalid_money)}')
print(f'Invalid Fraud      : {len(invalid_fraud)}')

Invalid Age        : 0
Invalid Quantity   : 0
Invalid Discount   : 0
Invalid Rating     : 0
Invalid Delivery   : 0
Invalid Money      : 0
Invalid Fraud      : 0


### 6. Save cleaned dataset

In [11]:
# Save the cleaned dataset to a new CSV file.
ecommerce_df.to_csv("dataset/cleaned/ecommerce_cleaned.csv", index=False)